In [1]:
import os
from openai import OpenAI
import env
import pandas as pd

In [2]:
pip install tabulate

Note: you may need to restart the kernel to use updated packages.


In [9]:
df = pd.read_csv("../data/restaurants.csv")
restaurant_context = df.to_string(index=False)

In [10]:
api_key = env.api_key

In [11]:
client = OpenAI(
    api_key = api_key,
    base_url="https://api.groq.com/openai/v1"
   
)

In [12]:
system_prompt = f"""
You are a helpful food assistant for LocalBuka. 
Your job is to help users find food based ONLY on the available dataset below. 
If a user asks for something not in the list, politely inform them.

CRITICAL FORMATTING RULES:
- Never respond with Markdown tables, pipe symbols (|), or JSON.
- Respond in natural, friendly prose using short bullet points or standard paragraphs.
- Keep recommendations clear, friendly, and brief.

Available Restaurant & Dish Data:
{restaurant_context}
"""

In [13]:
def get_bot_response(client, messages, user_input):
    messages.append({"role": "user", "content": user_input})

    completion = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
        max_tokens=1000,
        temperature=0.2,
        stream=True,
    )

    print("Assistant: ", end="")
    full_response = ""
    for chunk in completion:
        if chunk.choices[0].delta.content is not None:
            content = chunk.choices[0].delta.content
            print(content, end="")
            full_response += content
    print("\n")

    messages.append({"role": "assistant", "content": full_response})
    return full_response

In [14]:
messages = [{"role": "system", "content": system_prompt}]
print("LocalBuka Chatbot here! (Type 'quit' or 'exit' to stop)\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", "exit"]:
        print("Goodbye!")
        break
    
    get_bot_response(client, messages, user_input)

LocalBuka Chatbot here! (Type 'quit' or 'exit' to stop)



You:  i need vegetarian food


Assistant: Here are a few spots that serve vegetarian dishes and are easy to find:

- **Exodus** – Island breakfast place with a relaxed outdoor patio. Try their veggie‑filled pancakes or a hearty brunch.
- **Crimp’s Pizza** – Fast‑food pizza joint on the island. They offer a vegetarian pizza option, plus cocktails if you’re in the mood for something extra.
- **Spice Bistro** – Intercontinental eatery on the mainland. Their menu features vegetarian pasta dishes and a casual, outdoor seating area.
- **The Morning Griddle** – Breakfast spot on the mainland. Vegetarian waffles and omelettes are a hit, and they’re great for a quick coffee break.
- **Sunrise Pastries & Diner** – Island breakfast café. Vegetarian pastries, croissants, and coffee make for a light, tasty start to the day.
- **Wok & Roll** – Asian fusion on the island. They serve vegetarian noodles and stir‑fries, and you can order delivery if you prefer to stay in.
- **Cold Treats Corner** – Premium dessert



You:  thanks, exit


Assistant: You’re welcome! If you need anything else later, just let me know. Have a great day!



You:  exit


Goodbye!
